In [ ]:
# Inferentia 2 setup
# AWS Neuron-DLAMI ships /opt/aws_neuronx_venv_pytorch_2_9 with torch + torch_xla + torch_neuronx ready.
# This cell is intentionally a no-op on inf2; if you are running on a custom env, install:
#   pip install torch-neuronx neuronx-cc transformers
import os, sys
print("python:", sys.executable)
print("Inferentia setup: skipping pip install (assumes Neuron venv is already active)")


In [ ]:
import os, torch
print("torch:", torch.__version__)
try:
    import torch_neuronx
    print("torch_neuronx:", torch_neuronx.__version__)
except Exception as e:
    print("torch_neuronx import failed:", e)
try:
    import torch_xla
    print("torch_xla:", torch_xla.__version__)
except Exception as e:
    print("torch_xla import failed:", e)

# Required for neuronx-cc to preserve xp.Trace tags as framework_op_name in the NEFF.
# Without these, instruction[].layer in neuron-profile output is empty.
os.environ["XLA_HLO_DEBUG"] = "1"
os.environ["NEURON_FRAMEWORK_DEBUG"] = "1"
os.environ.setdefault("NEURON_CC_FLAGS", "--cache_dir=/var/tmp/neuron-compile-cache")
print("env XLA_HLO_DEBUG=", os.environ["XLA_HLO_DEBUG"])
print("env NEURON_FRAMEWORK_DEBUG=", os.environ["NEURON_FRAMEWORK_DEBUG"])
print("env NEURON_CC_FLAGS=", os.environ["NEURON_CC_FLAGS"])


In [ ]:
# No Google Drive mount on inf2. CSV writes to local OUTPUT_DIR (next cell).
print("inf2: no Drive mount needed")


In [ ]:
import os
OUTPUT_DIR = os.path.expanduser("~/LLMServingSim/profiler/perf")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output dir:", OUTPUT_DIR)


In [ ]:
# ====================== Inf2 micro-profiler core (inline) ======================
# Port of the TPU full-model + xp.Trace methodology to AWS Inferentia 2.
#
# Methodology preserved verbatim:
#   - Full HF model loaded, truncated to 1 layer
#   - Every sub-layer's forward monkey-patched with xla_timed_wrapper(tag, ...)
#   - Tags survive XLA -> HLO -> NEFF lowering (XLA_HLO_DEBUG=1) and appear in
#     neuron-profile JSON as instruction[].layer / layer_summary[].name
#   - Per-tag exclusive device time = sum of (tensor + scalar + vector + sync)
#     engine_active_time for all layer_summary entries whose name contains the tag
#
# What changed vs TPU:
#   - Capture mechanism: lazy XLA + xp.start_trace -> torch_neuronx.trace (AOT NEFF) +
#     subprocess(neuron-profile capture/view) per inference iteration
#   - Trace parser: Chrome trace JSON _exclusive_total -> Inf2 layer_summary tag-prefix sum
#   - Time unit: layer_summary fields are nanoseconds (verified against summary.total_time)
import os, gc, csv, time, math, types, json, gzip, statistics, subprocess, shutil, tempfile
from typing import List
from collections import defaultdict, namedtuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

import torch_neuronx
import torch_xla
import torch_xla.debug.profiler as xp


# -------- Lazy imports per-arch to avoid ImportError --------
def _llama_modules():
    import transformers.models.llama.modeling_llama as llm
    from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding, DynamicCache
    return llm, LlamaRotaryEmbedding, DynamicCache

def _phimoe_modules():
    import transformers.models.phimoe.modeling_phimoe as pm
    from transformers.models.phimoe.modeling_phimoe import PhimoeRotaryEmbedding
    return pm, PhimoeRotaryEmbedding


# -------- Tag wrapper (UNCHANGED from TPU notebook) --------
# xp.Trace context tag survives into NEFF metadata via XLA_HLO_DEBUG=1.
def xla_timed_wrapper(tag, fn, use_xla: bool = False):
    def wrapped(*args, **kwargs):
        if not use_xla:
            with torch.autograd.profiler.record_function(tag):
                return fn(*args, **kwargs)
        else:
            with xp.Trace(tag):
                out = fn(*args, **kwargs)
                return out
    return wrapped


# -------- KV cache builders (UNCHANGED) --------
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding, DynamicCache
def create_llama_past_key_values(config, kv_len, device):
    num_layers = config.num_hidden_layers
    num_kv = getattr(config, "num_key_value_heads", config.num_attention_heads)
    head_dim = config.hidden_size // config.num_attention_heads
    dtype = torch.bfloat16 if getattr(config, "torch_dtype", None) in (torch.bfloat16, None) else config.torch_dtype

    key_states = torch.zeros((1, num_kv, kv_len, head_dim), device=device, dtype=dtype)
    value_states = torch.zeros((1, num_kv, kv_len, head_dim), device=device, dtype=dtype)

    rope = LlamaRotaryEmbedding(config=config, device=device)

    dummy_x = torch.zeros((1, kv_len, head_dim), device=device, dtype=dtype)
    position_ids = torch.arange(kv_len, device=device).unsqueeze(0)
    cos, sin = rope(dummy_x, position_ids)

    cache = DynamicCache()
    for layer_idx in range(num_layers):
        cache.update(key_states, value_states, layer_idx, {
            "cos": cos, "sin": sin, "cache_position": position_ids
        })
    return cache

def create_phimoe_past_key_values(config, kv_len, device):
    from transformers.models.llama.modeling_llama import DynamicCache
    num_layers = config.num_hidden_layers
    num_kv = getattr(config, "num_key_value_heads", config.num_attention_heads)
    head_dim = config.hidden_size // config.num_attention_heads
    dtype = torch.bfloat16 if getattr(config, "torch_dtype", None) in (torch.bfloat16, None) else config.torch_dtype

    key_states = torch.zeros((1, num_kv, kv_len, head_dim), device=device, dtype=dtype)
    value_states = torch.zeros((1, num_kv, kv_len, head_dim), device=device, dtype=dtype)

    pm, PhimoeRotaryEmbedding = _phimoe_modules()
    rope = PhimoeRotaryEmbedding(config=config)

    dummy_x = torch.zeros((1, kv_len, head_dim), device=device, dtype=dtype)
    cos, sin = rope(dummy_x, kv_len)
    position_ids = torch.arange(kv_len, device=device).unsqueeze(0)

    cache = DynamicCache()
    for layer_idx in range(num_layers):
        cache.update(key_states, value_states, layer_idx, {
            "cos": cos, "sin": sin, "cache_position": position_ids
        })
    return cache


# -------- Arch-specific patching (UNCHANGED from TPU notebook) --------
def patch_llama_decoder_layer(layer, use_xla=False):
    sa, mlp = layer.self_attn, layer.mlp
    sa.q_proj.forward = xla_timed_wrapper("self_attn/q_proj", sa.q_proj.forward, use_xla)
    sa.k_proj.forward = xla_timed_wrapper("self_attn/k_proj", sa.k_proj.forward, use_xla)
    sa.v_proj.forward = xla_timed_wrapper("self_attn/v_proj", sa.v_proj.forward, use_xla)
    sa.o_proj.forward = xla_timed_wrapper("self_attn/o_proj", sa.o_proj.forward, use_xla)
    sa.forward = xla_timed_wrapper("self_attn", sa.forward, use_xla)

    mlp.gate_proj.forward = xla_timed_wrapper("mlp/gate_proj", mlp.gate_proj.forward, use_xla)
    mlp.up_proj.forward   = xla_timed_wrapper("mlp/up_proj",   mlp.up_proj.forward,   use_xla)
    mlp.down_proj.forward = xla_timed_wrapper("mlp/down_proj", mlp.down_proj.forward, use_xla)
    mlp.forward = xla_timed_wrapper("mlp", mlp.forward, use_xla)

    layer.input_layernorm.forward = xla_timed_wrapper("input_layernorm", layer.input_layernorm.forward, use_xla)
    layer.post_attention_layernorm.forward = xla_timed_wrapper("post_layernorm",  layer.post_attention_layernorm.forward, use_xla)


def patch_opt_decoder_layer(layer, use_xla: bool = False):
    sa = layer.self_attn
    sa.q_proj.forward   = xla_timed_wrapper("self_attn/q_proj", sa.q_proj.forward, use_xla)
    sa.k_proj.forward   = xla_timed_wrapper("self_attn/k_proj", sa.k_proj.forward, use_xla)
    sa.v_proj.forward   = xla_timed_wrapper("self_attn/v_proj", sa.v_proj.forward, use_xla)
    sa.out_proj.forward = xla_timed_wrapper("self_attn/o_proj", sa.out_proj.forward, use_xla)
    sa.forward = xla_timed_wrapper("self_attn", sa.forward, use_xla)
    layer.fc1.forward  = xla_timed_wrapper("mlp/fc1", layer.fc1.forward,  use_xla)
    layer.activation_fn = xla_timed_wrapper("mlp/act_fn", layer.activation_fn, use_xla)
    layer.fc2.forward = xla_timed_wrapper("mlp/fc2", layer.fc2.forward,  use_xla)
    layer.self_attn_layer_norm.forward = xla_timed_wrapper("input_layernorm", layer.self_attn_layer_norm.forward, use_xla)
    layer.final_layer_norm.forward = xla_timed_wrapper("post_layernorm",  layer.final_layer_norm.forward,     use_xla)


def patch_phimoe_decoder_layer(layer, use_xla=False):
    sa = layer.self_attn
    sa.q_proj.forward = xla_timed_wrapper("self_attn/q_proj", sa.q_proj.forward, use_xla)
    sa.k_proj.forward = xla_timed_wrapper("self_attn/k_proj", sa.k_proj.forward, use_xla)
    sa.v_proj.forward = xla_timed_wrapper("self_attn/v_proj", sa.v_proj.forward, use_xla)
    sa.o_proj.forward = xla_timed_wrapper("self_attn/o_proj", sa.o_proj.forward, use_xla)
    sa.forward = xla_timed_wrapper("self_attn", sa.forward, use_xla)

    moe = getattr(layer, "block_sparse_moe", None)
    if moe is not None:
        moe.forward = xla_timed_wrapper("mlp", moe.forward, use_xla)
        if hasattr(moe, "gate"):
            moe.gate.forward = xla_timed_wrapper("mlp/gate", moe.gate.forward, use_xla)
        try:
            pm, _ = _phimoe_modules()
        except NameError:
            pm = None
        if pm is not None and hasattr(pm, "sparsemixer"):
            pm.sparsemixer = xla_timed_wrapper("mlp/sparsemixer", pm.sparsemixer, use_xla)
        experts = getattr(moe, "experts", None)
        if experts is not None:
            for expert in experts:
                if hasattr(expert, "w1"):
                    expert.w1.forward = xla_timed_wrapper("mlp/expert.w1",   expert.w1.forward, use_xla)
                if hasattr(expert, "w3"):
                    expert.w3.forward = xla_timed_wrapper("mlp/expert.w2", expert.w3.forward, use_xla)
                if hasattr(expert, "w2"):
                    expert.w2.forward = xla_timed_wrapper("mlp/expert.w3", expert.w2.forward, use_xla)

    layer.input_layernorm.forward = xla_timed_wrapper("input_layernorm", layer.input_layernorm.forward, use_xla)
    layer.post_attention_layernorm.forward = xla_timed_wrapper("post_layernorm",  layer.post_attention_layernorm.forward, use_xla)


def patch_model(model, config, use_xla=False):
    archs = [a.lower() for a in getattr(config, "architectures", [])]
    arch = archs[0] if archs else ""
    if "llama" in arch:
        for lyr in model.model.layers:
            patch_llama_decoder_layer(lyr, use_xla=use_xla)
        model.model.embed_tokens.forward = xla_timed_wrapper("embedding", model.model.embed_tokens.forward, use_xla)
        model.model.norm.forward         = xla_timed_wrapper("final_layernorm", model.model.norm.forward, use_xla)
    elif "opt" in arch:
        for lyr in model.model.decoder.layers:
            patch_opt_decoder_layer(lyr, use_xla=use_xla)
        model.model.decoder.embed_tokens.forward     = xla_timed_wrapper("embedding", model.model.decoder.embed_tokens.forward, use_xla)
        model.model.decoder.final_layer_norm.forward = xla_timed_wrapper("final_layernorm", model.model.decoder.final_layer_norm.forward, use_xla)
    elif "phimoe" in arch:
        for lyr in model.model.layers:
            patch_phimoe_decoder_layer(lyr, use_xla=use_xla)
        model.model.embed_tokens.forward = xla_timed_wrapper("embedding", model.model.embed_tokens.forward, use_xla)
        model.model.norm.forward         = xla_timed_wrapper("final_layernorm", model.model.norm.forward, use_xla)
    else:
        raise NotImplementedError(f"Unsupported arch: {archs}")

    model.lm_head.forward = xla_timed_wrapper("lm_head", model.lm_head.forward, use_xla)


def _sanitize_model_name(name: str) -> str:
    return name.replace("/", "_").replace(":", "-")


# -------- Inf2 NEFF + neuron-profile capture/view helpers --------
NEURON_PROFILE_BIN = "/opt/aws/neuron/bin/neuron-profile"


class _ProfWrap(torch.nn.Module):
    """Wrap HF model so torch_neuronx.trace gets a clean (input_ids,) signature.
    KV cache is constructed inside if needed (fixed kv_len per trace)."""
    def __init__(self, base_model, kv_len: int):
        super().__init__()
        self.m = base_model
        self.kv_len = kv_len

    def forward(self, input_ids):
        if self.kv_len > 0:
            arch = (self.m.config.architectures[0].lower() if self.m.config.architectures else "")
            if "phi" in arch:
                pkv = create_phimoe_past_key_values(self.m.config, self.kv_len, input_ids.device)
            else:
                pkv = create_llama_past_key_values(self.m.config, self.kv_len, input_ids.device)
            out = self.m(input_ids=input_ids, past_key_values=pkv, use_cache=True)
        else:
            out = self.m(input_ids=input_ids, use_cache=True)
        return out.logits


def _trace_neff(model, input_len, kv_len, vocab_size, workdir):
    """Trace model -> NEFF. Returns NEFF path."""
    os.makedirs(workdir, exist_ok=True)
    example_ids = torch.randint(0, vocab_size, (1, input_len), dtype=torch.long)
    wrap = _ProfWrap(model, kv_len)
    traced = torch_neuronx.trace(wrap, (example_ids,), compiler_workdir=workdir)
    neff = os.path.join(workdir, "graph.neff")
    if not os.path.exists(neff):
        # fall back: search workdir
        for root, _, files in os.walk(workdir):
            for fn in files:
                if fn.endswith(".neff"):
                    neff = os.path.join(root, fn)
                    break
    return traced, neff


def _capture_and_view(neff_path, ntff_path, json_path, nth_exec=20, timeout_s=600):
    """Run neuron-profile capture (N inferences inside one process, dumps Nth)
    + view. Returns parsed JSON dict (or None on error).

    Why --profile-nth-exec: each `neuron-profile capture` invocation has ~30s of
    host overhead (NEFF load to HBM + runtime init + OpenAPI server boot) but
    the actual NEFF inference is ~6ms. Running N inferences in one capture
    amortizes the host overhead: 30s + N*6ms total instead of N*30s.

    Per-iter cost stays unchanged in the resulting NTFF (it dumps the Nth
    inference's instruction trace). NEFF execution is deterministic so a single
    snapshot is representative.
    """
    os.makedirs(os.path.dirname(ntff_path), exist_ok=True)
    cap_cmd = [
        NEURON_PROFILE_BIN, "capture",
        "-n", neff_path, "-s", ntff_path,
        f"--profile-nth-exec={nth_exec}",
    ]
    cap = subprocess.run(cap_cmd, capture_output=True, text=True, timeout=timeout_s)
    if cap.returncode != 0:
        print(f"[capture FAIL] rc={cap.returncode}  ntff={ntff_path}")
        print(f"  cmd: {' '.join(cap_cmd)}")
        print(f"  stderr (last 600c): {cap.stderr[-600:]}")
        print(f"  stdout (last 300c): {cap.stdout[-300:]}")
        return None
    if not os.path.exists(ntff_path) or os.path.getsize(ntff_path) == 0:
        print(f"[capture] rc=0 but NTFF missing/empty: {ntff_path}")
        print(f"  stderr (last 400c): {cap.stderr[-400:]}")
        return None

    view_cmd = [
        NEURON_PROFILE_BIN, "view",
        "--output-format", "json",
        "--output-file", json_path,
        "-n", neff_path, "-s", ntff_path,
    ]
    view = subprocess.run(view_cmd, capture_output=True, text=True, timeout=timeout_s)
    if view.returncode != 0:
        print(f"[view FAIL] rc={view.returncode}  json={json_path}")
        print(f"  cmd: {' '.join(view_cmd)}")
        print(f"  stderr (last 600c): {view.stderr[-600:]}")
        return None
    if not os.path.exists(json_path) or os.path.getsize(json_path) == 0:
        print(f"[view] rc=0 but JSON missing/empty: {json_path}")
        return None

    try:
        return json.load(open(json_path))
    except Exception as e:
        print(f"[view] JSON parse FAIL: {type(e).__name__}: {e}  path={json_path}")
        return None


# Tag list mirrors xla_timed_wrapper labels in patch_*_decoder_layer above.
_INF2_TAGS = [
    "embedding", "final_layernorm", "lm_head",
    "input_layernorm", "post_layernorm",
    "self_attn/q_proj", "self_attn/k_proj", "self_attn/v_proj", "self_attn/o_proj",
    "self_attn",
    "mlp/gate_proj", "mlp/up_proj", "mlp/down_proj",
    "mlp/fc1", "mlp/act_fn", "mlp/fc2",
    "mlp/gate", "mlp/sparsemixer",
    "mlp/expert.w1", "mlp/expert.w2", "mlp/expert.w3",
    "mlp",
]


def _aggregate_layer_summary(profile_json, tags=_INF2_TAGS):
    """Sum 4-engine active_time per tag prefix. Returns dict[tag] = nanoseconds.

    Each entry in profile_json['layer_summary'] has a name like
    '/sg00/self_attn/q_proj.9/aten__mm_dot.188' that contains our xp.Trace tag
    as a sub-path. Multiple entries may share the same tag (different aten ops).
    `tensor + scalar + vector + sync engine_active_time` is the device-side
    union duration attributable to that layer (engine-level overlap-resolved),
    which is the closest Inf2 analogue to the TPU notebook's _exclusive_total.
    """
    ls = profile_json.get("layer_summary", [])
    out = defaultdict(float)
    for entry in ls:
        name = entry.get("name", "")
        for tag in tags:
            # Match as a path component (avoid 'mlp' matching 'mlp/...' falsely)
            # We want longer tags to match first; resolve at the end.
            if tag in name:
                out[tag] += (
                    entry.get("tensor_engine_active_time", 0)
                    + entry.get("scalar_engine_active_time", 0)
                    + entry.get("vector_engine_active_time", 0)
                    + entry.get("sync_engine_active_time", 0)
                )
    return dict(out)


def run_profile(
    hardware="Inferentia2",
    model_name="meta-llama/Llama-3.2-1B-Instruct",
    num_layers=1,
    input_lengths=(128, 256),
    kv_cache_lengths=(0, 128),
    device_flag="cpu",     # legacy/compat: trace examples always live on CPU on Inf2
    warmup=0,              # unused on Inf2 — NEFF execution is deterministic
    nth_exec=20,           # # of inferences inside ONE capture; the Nth is dumped.
                           # Replaces TPU's "repeat=N forwards in one xp.start_trace"
                           # — both amortize host overhead, only Nth event window
                           # is preserved. NEFF deterministic so 1 snapshot OK.
    csv_append=True,
    verbose=True,
    out_dir="~/LLMServingSim/profiler/perf",
    hf_token="",
    progress=True,
    flush_every=4,
    neff_root="/tmp/inf2_neff",
    ntff_root="/tmp/inf2_ntff",
):
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
    out_dir = os.path.expanduser(out_dir)

    # 0) path & header
    out_dir_hw = os.path.join(out_dir, hardware)
    os.makedirs(out_dir_hw, exist_ok=True)
    out_path = os.path.join(out_dir_hw, f"{_sanitize_model_name(model_name)}.csv")
    fieldnames = ["hardware", "model", "layer_name", "input", "kv_cache", "latency(ns)", "compile(ns)"]

    def _ensure_header(path, overwrite=False):
        if overwrite:
            with open(path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=fieldnames).writeheader()
            return
        if (not os.path.exists(path)) or (os.path.getsize(path) == 0):
            with open(path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=fieldnames).writeheader()

    _ensure_header(out_path, overwrite=(not csv_append))
    if csv_append and (not os.path.exists(out_path) or os.path.getsize(out_path) == 0):
        _ensure_header(out_path, overwrite=False)

    # 1) model load (CPU — torch_neuronx.trace will JIT-lower to NEFF)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, token=hf_token or None)
    dtype = torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, token=hf_token or None).eval()
    original_num_layers = model.config.num_hidden_layers
    if num_layers is not None:
        if hasattr(model, "model") and hasattr(model.model, "layers"):
            model.model.layers = model.model.layers[:num_layers]
        elif hasattr(model, "model") and hasattr(model.model, "decoder"):
            model.model.decoder.layers = model.model.decoder.layers[:num_layers]
        model.config.num_hidden_layers = num_layers

    # IMPORTANT: use_xla=True ensures xp.Trace context is wrapped around every
    # sub-layer forward. The tag survives XLA->HLO->NEFF lowering because
    # XLA_HLO_DEBUG=1 (set in the env-cell at the top of the notebook).
    patch_model(model, model.config, use_xla=True)

    # 2) (input, kv) pairs
    pairs = []
    for il in input_lengths:
        if il <= 0: il = 1
        pairs.append((il, 0))
    for kl in kv_cache_lengths:
        if kl <= 0: kl = 1
        pairs.append((1, kl))

    outer = tqdm(pairs, disable=not progress, desc="Profiling configs", unit="cfg")
    buffered_rows = []
    since_last_flush = 0

    def _map_tags_to_results(exclusive_us: dict, arch: str):
        """Map our tags to CSV naming convention (UNCHANGED from TPU notebook)."""
        ex = exclusive_us
        out = {}
        if "embedding" in ex:          out["embedding"] = ex["embedding"]
        if "final_layernorm" in ex:    out["final_layernorm"] = ex["final_layernorm"]
        if "lm_head" in ex:            out["lm_head"] = ex["lm_head"]
        if "input_layernorm" in ex:    out["input_layernorm"] = ex["input_layernorm"]
        if "post_layernorm" in ex:     out["post_layernorm"] = ex["post_layernorm"]

        if "self_attn/q_proj" in ex: out["q_proj"] = ex["self_attn/q_proj"]
        if "self_attn/k_proj" in ex: out["k_proj"] = ex["self_attn/k_proj"]
        if "self_attn/v_proj" in ex: out["v_proj"] = ex["self_attn/v_proj"]
        if "self_attn/o_proj" in ex: out["o_proj"] = ex["self_attn/o_proj"]
        if "self_attn" in ex:
            out["attn"] = ex["self_attn"]

        if "llama" in arch:
            if "mlp/gate_proj" in ex: out["gate_proj"] = ex["mlp/gate_proj"]
            if "mlp/up_proj"   in ex: out["up_proj"]   = ex["mlp/up_proj"]
            if "mlp/down_proj" in ex: out["down_proj"] = ex["mlp/down_proj"]
            if "mlp"   in ex: out["act_fn"]    = ex["mlp"]
        elif "opt" in arch:
            if "mlp/fc1"     in ex: out["fc1"]     = ex["mlp/fc1"]
            if "mlp/act_fn"  in ex: out["act_fn"]  = ex["mlp/act_fn"]
            if "mlp/fc2"     in ex: out["fc2"]     = ex["mlp/fc2"]
        elif "phi" in arch:
            if "mlp/gate"      in ex: out["gate"]        = ex["mlp/gate"]
            if "mlp/sparsemixer" in ex: out["sparsemixer"] = ex["mlp/sparsemixer"]
            if "mlp/expert.w1"   in ex: out["expert.w1"]   = ex["mlp/expert.w1"]
            if "mlp/expert.w2"   in ex: out["expert.w2"]   = ex["mlp/expert.w2"]
            if "mlp/expert.w3"   in ex: out["expert.w3"]   = ex["mlp/expert.w3"]
            if "mlp"   in ex: out["act_fn"]    = ex["mlp"]
        return out

    # 3) loop — per (input, kv) config: trace -> capture (1 process, N inferences) -> aggregate
    arch = (model.config.architectures[0].lower() if model.config.architectures else "")
    sweep_start_ns = time.perf_counter_ns()
    for input_len, kv_len in outer:
        if progress:
            outer.set_postfix_str(f"in={input_len}, kv={kv_len}")

        # --- Trace NEFF for this shape ---
        cfg_id = f"in{input_len}_kv{kv_len}"
        workdir = os.path.join(neff_root, _sanitize_model_name(model_name), cfg_id)
        # Wipe stale workdir to force a fresh compile only when needed.
        if os.path.exists(workdir) and not os.path.exists(os.path.join(workdir, "graph.neff")):
            shutil.rmtree(workdir, ignore_errors=True)

        # compile(ns) records the wallclock of torch_neuronx.trace for this
        # (input_len, kv_len) shape. Three regimes:
        #   - 0       : NEFF already on disk in workdir (no trace call)
        #   - ~1-5s   : NEFF not on disk but NEURON_CC_FLAGS cache hits — host
        #               overhead only (re-emits the cached NEFF into workdir)
        #   - 30s+    : true cold compile — NEFF cache miss
        neff_path = os.path.join(workdir, "graph.neff")
        compile_ns = 0
        if not os.path.exists(neff_path):
            if verbose:
                print(f"[trace] compiling NEFF for {cfg_id}...")
            try:
                _t0 = time.perf_counter_ns()
                _, neff_path = _trace_neff(model, input_len, kv_len,
                                            tokenizer.vocab_size, workdir)
                compile_ns = time.perf_counter_ns() - _t0
                if verbose:
                    regime = "cache hit" if compile_ns < 30e9 else "cold compile"
                    print(f"[trace] {cfg_id} compile={compile_ns/1e6:.1f} ms ({regime})")
            except Exception as e:
                print(f"[trace FAIL] {cfg_id}: {type(e).__name__}: {e}")
                continue
        else:
            if verbose:
                print(f"[trace] reusing NEFF on disk: {neff_path} (compile=0)")

        # --- Capture once: --profile-nth-exec=N runs N inferences in one process ---
        ntff = os.path.join(ntff_root, _sanitize_model_name(model_name), f"{cfg_id}.ntff")
        json_path = ntff.replace(".ntff", ".json")
        d = _capture_and_view(neff_path, ntff, json_path, nth_exec=nth_exec)
        if d is None:
            print(f"[skip] no profile data for {cfg_id} (capture or view failed — see error above)")
            continue

        agg = _aggregate_layer_summary(d)
        if not agg:
            ls_entries = d.get("layer_summary", [])
            sample_names = [x.get("name", "") for x in ls_entries[:5]]
            print(f"[warn] {cfg_id}: layer_summary returned {len(ls_entries)} entries but NO tags matched.")
            print(f"  sample names: {sample_names}")
            print(f"  expected tags (first 5): {_INF2_TAGS[:5]}")
            try:
                os.remove(ntff); os.remove(json_path)
            except Exception:
                pass
            continue

        # ns -> us (single snapshot — no averaging since 1 capture)
        results_us = {k: v / 1000.0 for k, v in agg.items()}

        # Cleanup successful NTFF/JSON (saves disk; remove this if you want them for debug)
        try:
            os.remove(ntff)
            os.remove(json_path)
        except Exception:
            pass

        # 4) map tags + write rows
        results = _map_tags_to_results(results_us, arch=arch)
        if not results:
            print(f"[warn] {cfg_id}: aggregate had {len(agg)} tags but _map_tags returned empty.")
            print(f"  agg keys: {list(agg.keys())[:10]}")
            print(f"  arch: {arch!r}")
            continue

        if "llama" in arch:
            block_comps = ["input_layernorm", "q_proj", "k_proj", "v_proj", "rope", "attn", "o_proj",
                           "post_layernorm", "gate_proj", "up_proj", "act_fn", "down_proj"]
        elif "phi" in arch:
            block_comps = ["input_layernorm", "q_proj", "k_proj", "v_proj", "rope", "attn", "o_proj",
                           "post_layernorm", "gate", "sparsemixer", "expert.w1", "expert.w2", "expert.w3"]
        else:
            block_comps = ["input_layernorm", "q_proj", "k_proj", "v_proj", "qk_matmul", "softmax", "sv_matmul", "o_proj",
                           "post_layernorm", "fc1", "act_fn", "fc2"]
        common_comps = ["embedding", "final_layernorm", "lm_head"]

        estimated_latency = 0
        block_latency = 0
        for comp in block_comps:
            if comp in results:
                block_latency += int(max(results[comp], 0.0) * 1000.0)
        for comp in common_comps:
            if comp in results:
                estimated_latency += int(max(results[comp], 0.0) * 1000.0)
        estimated_latency += block_latency * original_num_layers / num_layers
        if verbose:
            print(f"input_len: {input_len}, kv_len: {kv_len}, estimated:{estimated_latency} ns")

        for comp in block_comps + common_comps:
            if comp in results:
                row = {
                    "hardware": hardware,
                    "model": model_name,
                    "layer_name": comp,
                    "input": input_len,
                    "kv_cache": kv_len,
                    "latency(ns)": int(max(results[comp], 0.0) * 1000.0),
                    "compile(ns)": int(compile_ns),
                }
                buffered_rows.append(row)
                if verbose:
                    print(row)

        since_last_flush += 1
        if since_last_flush >= max(1, flush_every):
            with open(out_path, "a", newline="") as f:
                w = csv.DictWriter(f, fieldnames=fieldnames)
                w.writerows(buffered_rows)
                f.flush()
                try: os.fsync(f.fileno())
                except Exception: pass
            buffered_rows.clear()
            since_last_flush = 0

        gc.collect()

    if buffered_rows:
        with open(out_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writerows(buffered_rows)
            f.flush()

    sweep_wallclock_ns = time.perf_counter_ns() - sweep_start_ns
    n_cfg = len(pairs)
    print(f"\n[done] sweep wallclock: {sweep_wallclock_ns/1e9:.1f} s  "
          f"({n_cfg} configs, avg {sweep_wallclock_ns/1e9/max(1,n_cfg):.1f} s/cfg)")

    # Sweep-level metadata (sweep wallclock is the headline, not per-row compile_ns)
    meta_path = out_path + ".meta.json"
    try:
        with open(meta_path, "w") as f:
            json.dump({
                "hardware": hardware,
                "model_name": model_name,
                "num_layers": num_layers,
                "n_configs": n_cfg,
                "sweep_wallclock_ns": int(sweep_wallclock_ns),
                "sweep_wallclock_s_per_cfg": sweep_wallclock_ns / 1e9 / max(1, n_cfg),
                "nth_exec": nth_exec,
                "input_lengths": list(input_lengths),
                "kv_cache_lengths": list(kv_cache_lengths),
            }, f, indent=2)
        print(f"[OK] meta saved: {meta_path}")
    except Exception as e:
        print(f"[warn] could not write meta.json: {e}")

    print(f"[OK] CSV stored: {out_path}")
    return out_path


In [ ]:
# === Validation utilities — Inf2 reproduction of TPU notebook ===
# Mirror of TPU cell 5: measure_generation_latency / estimate_total_latency /
# validate_latency_estimation / scale_latency_csv. The only thing that changes
# is HOW we time a forward pass:
#   TPU:  lazy XLA + torch_xla.sync() + perf_counter
#   Inf2: torch_neuronx.trace per shape -> traced(input) + perf_counter
# Wallclock semantics are identical (host-side perf_counter around device call).
import os, csv, time, statistics
from collections import defaultdict
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig


def _csv_path(out_dir, hardware, model_name):
    return os.path.join(out_dir, hardware, f"{model_name.replace('/', '_').replace(':', '-')}.csv")


# Inf2-native equivalent of TPU's lazy XLA forward — AOT-compile a NEFF for the
# given (input_len, kv_len) shape, return a traced module that runs the full
# 1-layer-truncated HF model. Cache hit is via NEURON_CC_FLAGS=--cache_dir=...
# (set in cell 1). First compile per shape ~minutes; subsequent ~seconds.
def _trace_for_shape(model, input_len, kv_len, vocab_size, workdir):
    os.makedirs(workdir, exist_ok=True)
    example_ids = torch.randint(0, vocab_size, (1, input_len), dtype=torch.long)
    wrap = _ProfWrap(model, kv_len)
    return torch_neuronx.trace(wrap, (example_ids,), compiler_workdir=workdir)


def measure_generation_latency(
    model,
    input_length=10,
    output_length=5,
    warmup=3,
    repeat=5,
    verbose=False,
    vocab_size=None,
    model_name=None,
    cache_root="/tmp/inf2_validate_neff",
):
    """Inf2 reproduction of TPU measure_generation_latency.

    Compiles 1 prefill NEFF (input_length, kv=0) + (output_length-1) decode NEFFs
    (1, kv=input_length+i), then for each of `repeat` iterations runs the full
    prefill + decode-loop sequence under perf_counter and reports the median (ms).
    """
    if vocab_size is None:
        vocab_size = model.config.vocab_size

    cache_root = os.path.join(cache_root, _sanitize_model_name(model_name or "anon"))
    os.makedirs(cache_root, exist_ok=True)

    if verbose:
        print(f"[validate.measure] tracing prefill NEFF (input={input_length}, kv=0)...")
    prefill_traced = _trace_for_shape(
        model, input_length, 0, vocab_size,
        os.path.join(cache_root, f"prefill_in{input_length}"),
    )

    decode_traced = []
    for i in range(max(0, output_length - 1)):
        kv = input_length + i
        if verbose:
            print(f"[validate.measure] tracing decode NEFF (input=1, kv={kv})...")
        traced = _trace_for_shape(
            model, 1, kv, vocab_size,
            os.path.join(cache_root, f"decode_in1_kv{kv}"),
        )
        decode_traced.append(traced)

    prefill_input = torch.randint(0, vocab_size, (1, input_length), dtype=torch.long)
    decode_input = torch.randint(0, vocab_size, (1, 1), dtype=torch.long)

    # ---- Warmup ----
    for _ in range(warmup):
        _ = prefill_traced(prefill_input)
        for traced in decode_traced:
            _ = traced(decode_input)

    # ---- Measure: full prefill + decode loop, median over `repeat` ----
    total_ns_runs = []
    for _ in range(repeat):
        t0 = time.perf_counter_ns()
        _ = prefill_traced(prefill_input)
        for traced in decode_traced:
            _ = traced(decode_input)
        total_ns_runs.append(time.perf_counter_ns() - t0)

    m_ns = statistics.median(total_ns_runs)
    dt_ms = m_ns / 1e6
    if verbose:
        print(f"[measure(step)] input={input_length}, output={output_length} -> {dt_ms:.2f} ms")
    return dt_ms


def estimate_total_latency(
    out_dir,
    hardware,
    model_name="meta-llama/Llama-3.2-1B",
    num_layers=32,
    input_length=10,
    output_length=5,
    csv_path=None,
    verbose=False,
):
    """UNCHANGED from TPU notebook — sums per-op latencies from the profile CSV
    for the given (input_length, output_length) generation sequence."""
    config = AutoConfig.from_pretrained(model_name)

    latency_db = defaultdict(dict)
    csv_path = csv_path or _csv_path(out_dir, hardware, model_name)
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            key = (int(row["input"]), int(row["kv_cache"]))
            latency_db[key][row["layer_name"]] = int(row["latency(ns)"])

    arch = model_name.lower()
    if "llama" in arch:
        comps = ["input_layernorm", "q_proj", "k_proj", "v_proj", "rope", "attn", "o_proj",
                 "post_layernorm", "gate_proj", "up_proj", "act_fn", "down_proj"]
    elif "phi" in arch:
        comps = ["input_layernorm", "q_proj", "k_proj", "v_proj", "rope", "attn", "o_proj",
                 "post_layernorm", "gate", "sparsemixer"]
    else:
        comps = ["input_layernorm", "q_proj", "k_proj", "v_proj", "qk_matmul", "softmax", "sv_matmul",
                 "o_proj", "post_layernorm", "fc1", "act_fn", "fc2"]

    total_ns = 0

    pre_key = (input_length, 0)
    if pre_key not in latency_db:
        raise ValueError(f"Missing latency for input={input_length}, kv=0 in {csv_path}")
    pre = latency_db[pre_key]
    total_ns += num_layers * sum(pre.get(c, 0) for c in comps)
    for c in ["embedding", "final_layernorm", "lm_head"]:
        total_ns += pre.get(c, 0)

    if "phi" in arch:
        n_experts = getattr(config, "num_local_experts", 1)
        moe_comps = ["expert.w1", "expert.w2", "act_fn", "expert.w3"]
        total_ns += num_layers * n_experts * sum(pre.get(c, 0) for c in moe_comps)

    for i in range(max(0, output_length - 1)):
        kv = input_length + i
        dec_key = (1, kv)
        if dec_key not in latency_db:
            raise ValueError(f"Missing latency for input=1, kv={kv} in {csv_path}")
        dec = latency_db[dec_key]
        total_ns += num_layers * sum(dec.get(c, 0) for c in comps)
        total_ns += dec.get("lm_head", 0)
        if "phi" in arch:
            total_ns += num_layers * n_experts * sum(dec.get(c, 0) for c in moe_comps)
        for c in ["embedding", "final_layernorm", "lm_head"]:
            total_ns += dec.get(c, 0)
        # TPU notebook adds a 10 ms-per-step constant to model JIT compile cost
        # at decode time. Kept here for reproduction parity. Pure device-time
        # estimate would set this to 0 (Inf2 NEFFs are AOT compiled).
        total_ns += 10e6 * i

    if verbose:
        print(f"[estimate] {total_ns/1e6:.3f} ms (from CSV)")
    return total_ns / 1e6


def validate_latency_estimation(
    out_dir,
    hardware,
    model_name="meta-llama/Llama-3.2-1B",
    num_layers=None,
    input_lengths=(10, 20),
    output_lengths=(2, 4),
    warmup=3,
    repeat=10,
    verbose=False,
    csv_path=None,
):
    """For each (input, output) pair: measured wallclock vs CSV-summed estimate.
    Returns rows of (input, output, measured_ms, estimated_ms, signed_pct_err)."""
    dtype = torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).eval()

    full_layers = AutoConfig.from_pretrained(model_name).num_hidden_layers
    if num_layers is not None:
        if hasattr(model, "model") and hasattr(model.model, "layers"):
            model.model.layers = model.model.layers[:num_layers]
        elif hasattr(model, "model") and hasattr(model.model, "decoder"):
            model.model.decoder.layers = model.model.decoder.layers[:num_layers]
        model.config.num_hidden_layers = num_layers

    # Patch with the same tags used in run_profile so NEFFs hash-match (NEFF cache
    # hit on second compile for the same shape).
    patch_model(model, model.config, use_xla=True)

    rows = []
    for il in input_lengths:
        for ol in output_lengths:
            il = 1 if il <= 0 else il
            ol = 1 if ol <= 0 else ol
            m_meas = measure_generation_latency(
                model, il, ol,
                warmup=warmup, repeat=repeat,
                model_name=model_name, verbose=verbose,
            )
            m_est = estimate_total_latency(
                out_dir, hardware, model_name,
                num_layers or full_layers,
                il, ol, csv_path=csv_path, verbose=False,
            )
            err = (m_est - m_meas) / max(1e-9, m_meas) * 100.0
            rows.append((il, ol, m_meas, m_est, err))
            if verbose:
                print(f"[validate] in={il}, out={ol}  measured={m_meas:.2f} ms  estimated={m_est:.2f} ms  Δ={err:+.1f}%")
    return rows


def scale_latency_csv(out_dir, hardware, model_name, scaling_factor=1.0, overwrite=False):
    """UNCHANGED from TPU notebook — apply scalar to latency(ns) column."""
    src = _csv_path(out_dir, hardware, model_name)
    dst = src if overwrite else src.replace(".csv", f".scaled_{scaling_factor:.2f}.csv")

    with open(src, newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames.copy()
        rows = []
        if not overwrite and "scaled_latency(ns)" not in fieldnames:
            fieldnames.append("scaled_latency(ns)")
        for row in reader:
            val = int(row["latency(ns)"])
            if overwrite:
                row["latency(ns)"] = int(val * scaling_factor)
            else:
                row["scaled_latency(ns)"] = int(val * scaling_factor)
            rows.append(row)

    with open(dst, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)

    print(f"[OK] wrote: {dst}")
    return dst


In [ ]:
# === Inf2 reproduction of TPU validate_and_scale ===
# UNCHANGED logic from TPU notebook — the underlying measure/estimate functions
# in cell 5 are now Inf2-native.
from statistics import median


def _compute_scaling_factor(rows, method="median"):
    ratios = [meas / est for (_, _, meas, est, _) in rows if est and est > 0]
    if not ratios:
        return 1.0
    if method == "mean":
        return float(sum(ratios) / len(ratios))
    return float(median(ratios))


def _compute_error_stats(rows):
    errs = [err for (*_, err) in rows if err is not None]
    if not errs:
        return {"n": 0, "mean_signed_pct_err": 0.0,
                "mean_abs_pct_err": 0.0, "median_abs_pct_err": 0.0}
    abs_errs = [abs(e) for e in errs]
    return {
        "n": len(errs),
        "mean_signed_pct_err": float(sum(errs) / len(errs)),
        "mean_abs_pct_err": float(sum(abs_errs) / len(abs_errs)),
        "median_abs_pct_err": float(median(abs_errs)),
    }


def validate_and_scale(
    out_dir,
    hardware,
    model_name,
    num_layers=None,
    input_lengths=(1,),
    output_lengths=(1, 2),
    warmup=5,
    repeat=3,
    verbose=True,
    method="median",
    overwrite=False,
    csv_path=None,
    return_error_stats=False,
):
    rows = validate_latency_estimation(
        out_dir=out_dir, hardware=hardware, model_name=model_name,
        num_layers=num_layers,
        input_lengths=input_lengths, output_lengths=output_lengths,
        warmup=warmup, repeat=repeat, verbose=verbose, csv_path=csv_path,
    )

    sf = _compute_scaling_factor(rows, method=method)
    if verbose:
        print(f"[scale] fitted scaling_factor = {sf:.4f} (method={method})")

    err_stats = _compute_error_stats(rows)
    if verbose:
        print("[error] n={n}  mean_abs={ma:.2f}%  median_abs={md:.2f}%  mean_signed={ms:+.2f}%".format(
            n=err_stats["n"],
            ma=err_stats["mean_abs_pct_err"],
            md=err_stats["median_abs_pct_err"],
            ms=err_stats["mean_signed_pct_err"],
        ))

    scaled_path = scale_latency_csv(
        out_dir=out_dir, hardware=hardware, model_name=model_name,
        scaling_factor=sf, overwrite=overwrite,
    )
    if verbose:
        print(f"[scale] wrote scaled CSV: {scaled_path}")

    if return_error_stats:
        return sf, scaled_path, rows, err_stats
    return sf, scaled_path, rows


# **RUN PROFILE**

In [ ]:
# ====================== Profiling Parameters (edit here) ======================
HARDWARE = "Inferentia2"
MODEL    = "meta-llama/Llama-3.2-1B-Instruct"
NUM_LAYERS = 1            # truncate model to 1 decoder layer
DEVICE  = "cpu"           # legacy/compat — Inf2 always traces on CPU + runs NEFF on Neuron
NTH_EXEC = 20             # # of inferences inside ONE neuron-profile capture (the Nth is
                          # dumped). Replaces TPU's "repeat=N forwards in one trace window"
                          # — both amortize host overhead. NEFF deterministic so the Nth
                          # snapshot is representative.
PREFILL_MAX = 256         # sequence length sweep upper bound
PREFILL_STEP = 128
DECODE_MAX  = 256         # kv-cache length sweep upper bound
DECODE_STEP = 128
HF_TOKEN = ""             # set if model is gated

# Build sweeps (prefill includes 1; kv includes 0)
input_prefill = list(range(0, PREFILL_MAX + 1, PREFILL_STEP))
kv_decode    = list(range(0, DECODE_MAX + 1, DECODE_STEP))
print("input lens:", len(input_prefill), input_prefill)
print("kv lens:   ", len(kv_decode), kv_decode)
print(f"total configs: {len(input_prefill) + len(kv_decode)} (NEFF compile per config; cached after first run)")


In [ ]:
# ====================== Run profiling ======================
out_csv = run_profile(
    hardware=HARDWARE,
    model_name=MODEL,
    num_layers=NUM_LAYERS,
    input_lengths=tuple(input_prefill),
    kv_cache_lengths=tuple(kv_decode),
    device_flag=DEVICE,
    nth_exec=NTH_EXEC,
    csv_append=False,
    verbose=True,
    out_dir=OUTPUT_DIR,
    hf_token=HF_TOKEN,
)


In [ ]:
# ====================== Validation Parameters (edit here) ======================
# Validation works by replaying (input_length, kv_cache=input_length+i) from the
# profile CSV. Constraint: every (input, kv) pair touched during validation must
# exist in the CSV produced by Cell 9.
#
# With Cell 8's default sweep (PREFILL_STEP=128, DECODE_STEP=128, MAX=256), the
# CSV covers (input ∈ {1,128,256}, kv=0) and (input=1, kv ∈ {1,128,256}). So
# validation must restrict to those:
#   - input_lengths ⊆ {128, 256}    (prefill must hit one of those exact values)
#   - For output_length > 1, decode kv = input + i must also be in CSV's kv set.
#     With STEP=128 that means output_length must keep kv ∈ {128, 129, ..., 256+}
#     within profile range — impractical at STEP=128.
#
# Recommended: keep validation cheap with output_length=1 (prefill-only) for the
# initial pass, then narrow PREFILL_STEP/DECODE_STEP in Cell 8 and re-profile if
# you want a multi-step decode validation.
HARDWARE = "Inferentia2"
MODEL    = "meta-llama/Llama-3.2-1B-Instruct"
NUM_LAYERS = 1
WARMUP  = 3
REPEAT  = 5
INPUT_LENGTHS  = (128, 256)
OUTPUT_LENGTHS = (1,)        # prefill-only validation; bump to >1 only if CSV covers all kv steps

print(f"validation: HARDWARE={HARDWARE}, MODEL={MODEL}")
print(f"  input_lengths={INPUT_LENGTHS}, output_lengths={OUTPUT_LENGTHS}")
print(f"  warmup={WARMUP}, repeat={REPEAT}")
print()
print("Note: each (input, kv) pair touched here must exist in the profile CSV.")
print("If estimate raises 'Missing latency for input=X, kv=Y', narrow Cell 8's")
print("PREFILL_STEP / DECODE_STEP and re-run profiling to cover the gap.")


In [ ]:
# ====================== Run validation ======================
sf, scaled_csv, rows = validate_and_scale(
    out_dir=OUTPUT_DIR,
    hardware=HARDWARE,
    model_name=MODEL,
    num_layers=NUM_LAYERS,
    input_lengths=INPUT_LENGTHS,
    output_lengths=OUTPUT_LENGTHS,
    warmup=WARMUP,
    repeat=REPEAT,
    verbose=True,
    overwrite=False,
)
print(f"\n[done] scaling_factor = {sf:.4f}")
print(f"[done] scaled CSV     : {scaled_csv}")
print()
print("per-cfg detail:")
for il, ol, meas, est, err in rows:
    print(f"  in={il:>4}  out={ol:>3}  meas={meas:>9.2f} ms  est={est:>9.2f} ms  signed_err={err:+7.2f}%")


In [ ]:
# ====================== Test error rate after scaling ======================
# Apply the scaled CSV in place of the unscaled one (with backup), then
# re-validate to confirm the error rate has dropped.
import os, shutil, statistics

base = _csv_path(OUTPUT_DIR, HARDWARE, MODEL)
backup = base + ".unscaled.bak"

if not os.path.exists(backup):
    shutil.copy(base, backup)
    print(f"[backup] saved unscaled CSV -> {backup}")

shutil.copy(scaled_csv, base)
print(f"[apply] copied scaled CSV   -> {base}")

# Re-run validation (now estimate will use scaled latencies)
rows2 = validate_latency_estimation(
    out_dir=OUTPUT_DIR,
    hardware=HARDWARE,
    model_name=MODEL,
    num_layers=NUM_LAYERS,
    input_lengths=INPUT_LENGTHS,
    output_lengths=OUTPUT_LENGTHS,
    warmup=WARMUP,
    repeat=REPEAT,
    verbose=True,
)

print("\n=== error after scaling ===")
for il, ol, meas, est, err in rows2:
    print(f"  in={il:>4}  out={ol:>3}  meas={meas:>9.2f} ms  est={est:>9.2f} ms  signed_err={err:+7.2f}%")

abs_errs = [abs(e) for (_, _, _, _, e) in rows2]
if abs_errs:
    print(f"\n  mean_abs_err   = {statistics.fmean(abs_errs):.2f}%")
    print(f"  median_abs_err = {statistics.median(abs_errs):.2f}%")
    print(f"  max_abs_err    = {max(abs_errs):.2f}%")

print(f"\n[restore] to revert: cp {backup} {base}")
